# Évaluation et calibration du seuil

Ce notebook traite les `person_id` de `data/labels.csv` comme vérité terrain; les tokens des noms de fichiers ne définissent pas l'identité. Une image par identité évaluée sert de requête et est exclue des résultats. Les scores sur ce petit jeu ne sont pas représentatifs d'une performance générale.

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from src.config import load_config
from src.detection import FaceDetector
from src.embedding import InsightFaceEmbedder
from src.evaluation import evaluate, load_labels, roc_curve_pairs, sweep_threshold
from src.gallery import build_gallery_index
from src.io_utils import read_image
from src.matching import match_images
from src.identity import resolve_identity

project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
labels_path = project_root / "data" / "labels.csv"
gallery_path = project_root / "data" / "gallery"
config = load_config()
labels = load_labels(labels_path)

labelled_images = {filename for filename in labels}
gallery_images = {
    path.name
    for path in gallery_path.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
}
print(f"Images annotées : {len(labelled_images)} / images de galerie : {len(gallery_images)}")
print(f"Identités distinctes : {len({person_id for ids in labels.values() for person_id in ids})}")
print("Images sans annotation :", sorted(gallery_images - labelled_images))
print("Annotations sans image correspondante :", sorted(labelled_images - gallery_images))

embedder = InsightFaceEmbedder(config)
detector = FaceDetector(config, analysis=embedder.analysis)
records = build_gallery_index(
    gallery_path,
    detector,
    embedder,
    cache_path=config.paths.cache_dir / "embeddings.npz",
)
print(f"Visages indexés : {len(records)}")

## Balayage des seuils

Les courbes utilisent la moyenne macro par identité et 51 seuils entre 0,20 et 0,70. Le meilleur F1 est une recommandation à examiner, pas une mise à jour automatique de `configs/config.yaml`.

In [ ]:
threshold_results = sweep_threshold(records, labels_path)
display(threshold_results[:5])

best_row = max(
    threshold_results,
    key=lambda row: (
        row["f1"],
        row["precision"],
        -abs(row["threshold"] - config.matching.threshold),
        -row["threshold"],
    ),
)
print(
    f"Seuil suggéré : {best_row['threshold']:.2f} "
    f"(F1 macro={best_row['f1']:.3f}, précision={best_row['precision']:.3f}, "
    f"rappel={best_row['recall']:.3f}, requêtes={int(best_row['n_queries'])})"
)

fig, ax = plt.subplots(figsize=(9, 5))
for metric in ("precision", "recall", "f1"):
    ax.plot(
        [row["threshold"] for row in threshold_results],
        [row[metric] for row in threshold_results],
        label=metric.capitalize(),
    )
ax.axvline(best_row["threshold"], color="black", linestyle="--", label="Meilleur F1")
ax.set(xlabel="Seuil cosinus", ylabel="Moyenne macro", ylim=(0, 1.02))
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Comparaison des stratégies d'identité

À seuil identique, comparer la recherche de base, l'expansion de requête et le clustering. Une requête déterministe par identité est utilisée; les identités qui n'ont pas au moins deux images indexées ne peuvent pas contribuer au rappel.

In [ ]:
records_by_filename = {}
for record in records:
    records_by_filename.setdefault(Path(record.image_path).name, []).append(record)

query_samples = []
person_ids = sorted({person_id for ids in labels.values() for person_id in ids})
for person_id in person_ids:
    available = sorted(
        filename
        for filename, ids in labels.items()
        if person_id in ids and filename in records_by_filename
    )
    if len(available) >= 2:
        query_filename = available[0]
        query_record = records_by_filename[query_filename][0]
        query_samples.append((person_id, query_record.image_path, query_record.embedding))

strategy_options = {
    "Seuil seul": (False, False),
    "Expansion de requête": (True, False),
    "Clustering galerie": (False, True),
}
strategy_results = []
for strategy, (use_expansion, use_clustering) in strategy_options.items():
    strategy_config = replace(
        config,
        matching=replace(
            config.matching,
            threshold=float(best_row["threshold"]),
            use_query_expansion=use_expansion,
            use_gallery_clustering=use_clustering,
        ),
    )
    per_identity = [
        evaluate(
            resolve_identity(embedding, records, strategy_config),
            labels_path,
            person_id,
            query_path,
        )
        for person_id, query_path, embedding in query_samples
    ]
    strategy_results.append(
        {
            "strategie": strategy,
            "precision": float(np.mean([metric["precision"] for metric in per_identity])),
            "recall": float(np.mean([metric["recall"] for metric in per_identity])),
            "f1": float(np.mean([metric["f1"] for metric in per_identity])),
            "identites_evaluees": len(per_identity),
        }
    )
display(strategy_results)

## ROC et EER

Les paires sont construites entre les visages de deux images distinctes. Une paire est positive si les images partagent au moins un `person_id`.

In [ ]:
roc = roc_curve_pairs(records, labels_path)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(roc["fpr"], roc["tpr"], label=f"ROC (EER={roc['eer']:.3f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Aléatoire")
ax.set(xlabel="Taux de faux positifs", ylabel="Taux de vrais positifs", xlim=(0, 1), ylim=(0, 1))
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print(
    f"EER={roc['eer']:.3f} au seuil interpolé {roc['eer_threshold']:.3f}; "
    f"paires positives={roc['n_positive_pairs']}, négatives={roc['n_negative_pairs']}"
)

## Inspection des faux positifs et faux négatifs

Les exemples ci-dessous utilisent le seuil suggéré et la même règle de sélection déterministe des requêtes que le balayage.

In [ ]:
records_by_filename = {}
for record in records:
    records_by_filename.setdefault(Path(record.image_path).name, []).append(record)

false_positives = []
false_negatives = []
person_ids = sorted({person_id for ids in labels.values() for person_id in ids})
for person_id in person_ids:
    available = sorted(
        filename
        for filename, ids in labels.items()
        if person_id in ids and filename in records_by_filename
    )
    if len(available) < 2:
        continue
    query_filename = available[0]
    query_record = records_by_filename[query_filename][0]
    predictions = {
        Path(match.image_path).name
        for match in match_images(query_record.embedding, records, float(best_row["threshold"]))
        if Path(match.image_path).name != query_filename
    }
    expected = {
        filename
        for filename, ids in labels.items()
        if person_id in ids and filename != query_filename
    }
    false_positives.extend((person_id, filename) for filename in sorted(predictions - expected))
    false_negatives.extend((person_id, filename) for filename in sorted(expected - predictions))

print("Faux positifs :", false_positives[:10])
print("Faux négatifs :", false_negatives[:10])

errors = [("Faux positif", person_id, filename) for person_id, filename in false_positives[:4]]
errors += [("Faux négatif", person_id, filename) for person_id, filename in false_negatives[:4]]
if errors:
    fig, axes = plt.subplots(len(errors), 1, figsize=(7, 3 * len(errors)))
    axes = np.atleast_1d(axes)
    for ax, (kind, person_id, filename) in zip(axes, errors, strict=True):
        image = read_image(gallery_path / filename)
        ax.imshow(image[:, :, ::-1])
        ax.set_title(f"{kind} — identité {person_id} — {filename}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Aucune erreur à afficher pour les identités évaluables.")

## Décision de calibration

La configuration du projet conserve `matching.threshold: 0.40`, un seuil situé sur le plateau optimal mesuré. Les résultats sont parfaits sur ce petit jeu; ils doivent être interprétés avec prudence et ne démontrent pas une performance générale.

## Résultats observés

- 29 images annotées; 10 identités avec au moins deux images indexées utilisées comme requêtes.
- Sur les 51 seuils de 0,20 à 0,70 : précision macro = 1,00, rappel macro = 1,00 et F1 macro = 1,00. Le seuil existant 0,40 est conservé parmi les valeurs ex æquo.
- ROC : EER = 0,00 sur 12 paires positives et 394 paires négatives.
- À 0,40, seuil seul, expansion de requête et clustering galerie ont chacun P/R/F1 = 1,00.